# 🏦 Bank Cash Optimization — Phase 5: XAI (Explainable AI)

---

**Project:** Bank Cash Optimization  
**Notebook:** 05_XAI_Explainability  
**Author:** Abdul Qadeer  

## 🎯 Objective

Use **SHAP (SHapley Additive exPlanations)** to explain WHY the model predicts a certain cash demand.

**This notebook covers:**
1. Load best trained model
2. Create SHAP explainer
3. Global Explanation — which features matter most overall?
4. SHAP Summary Plot (bar + beeswarm)
5. SHAP Waterfall Plot — explain one specific prediction
6. SHAP Dependence Plot — how lag_1 affects prediction
7. Branch-level explanation — why Branch X needs this much cash

> **Business Value:** Manager ko pata chalega ki model kis wajah se keh raha hai  
> 'Branch X ko kal 5 Crore chahiye' — kyunki Monday hai + salary week + high lag_1


---
## Step 1 — Import Libraries

In [1]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import shap
import joblib, json, os

pd.set_option('display.float_format', '{:,.2f}'.format)
os.makedirs('eda_plots', exist_ok=True)

plt.rcParams.update({
    'figure.dpi'       : 130,
    'axes.titlesize'   : 13,
    'axes.titleweight' : 'bold',
    'axes.labelsize'   : 11,
    'axes.spines.top'  : False,
    'axes.spines.right': False,
    'figure.facecolor' : 'white',
})

print('✅ Libraries imported (SHAP version:', shap.__version__, ')')

✅ Libraries imported (SHAP version: 0.52.0 )


---
## Step 2 — Load Model & Data

In [2]:
# Load best model
model = joblib.load('models/best_model.pkl')

# Load features
X_train = pd.read_csv('model_data/X_train.csv')
X_test  = pd.read_csv('model_data/X_test.csv')
y_test  = pd.read_csv('model_data/y_test.csv').squeeze()

with open('model_data/feature_cols.json') as f:
    feature_cols = json.load(f)

# Load daily data for branch info
daily = pd.read_csv('model_data/daily_full.csv')
daily['start_date'] = pd.to_datetime(daily['start_date'])
daily = daily.sort_values('start_date').reset_index(drop=True)
cutoff_date = daily.iloc[int(len(daily)*0.80)]['start_date']
test_meta   = daily[daily['start_date'] >= cutoff_date].reset_index(drop=True)

print(f'✅ Model loaded: {type(model).__name__}')
print(f'✅ X_train: {X_train.shape}  |  X_test: {X_test.shape}')
print(f'✅ Features: {len(feature_cols)}')

✅ Model loaded: XGBRegressor
✅ X_train: (6851, 22)  |  X_test: (1731, 22)
✅ Features: 22


---
## Step 3 — Create SHAP Explainer

SHAP TreeExplainer is optimized for XGBoost/LightGBM tree-based models.

In [3]:
print('🔄 Creating SHAP TreeExplainer...')
explainer   = shap.TreeExplainer(model)

# Compute SHAP values for test set (sample 500 for speed)
sample_size = min(500, len(X_test))
X_sample    = X_test.iloc[:sample_size].reset_index(drop=True)

print(f'🔄 Computing SHAP values for {sample_size} test samples...')
shap_values = explainer.shap_values(X_sample)

# Also get full explanation object
explanation = explainer(X_sample)

print(f'✅ SHAP values computed!')
print(f'   Shape: {shap_values.shape}  (rows × features)')
print(f'   Expected value (base): PKR {explainer.expected_value/1e6:.2f} Million')

🔄 Creating SHAP TreeExplainer...


🔄 Computing SHAP values for 500 test samples...


✅ SHAP values computed!
   Shape: (500, 22)  (rows × features)
   Expected value (base): PKR 46.34 Million


---
## Step 4 — Global Explanation: SHAP Summary Bar Plot

**Konsa feature overall sab se zyada important hai?**

In [4]:
fig, ax = plt.subplots(figsize=(10, 7))
shap.summary_plot(
    shap_values, X_sample,
    plot_type='bar',
    feature_names=feature_cols,
    show=False,
    max_display=22
)
plt.title('SHAP Global Feature Importance \u2014 Which Features Drive Cash Predictions Most?',
          fontsize=12, fontweight='bold', pad=15)
plt.xlabel('Mean |SHAP Value| (Impact on prediction in PKR)')
plt.tight_layout()
plt.savefig('eda_plots/13_shap_global_importance.png', dpi=150, bbox_inches='tight')
plt.close()
print('✅ SHAP Global Importance plot saved → eda_plots/13_shap_global_importance.png')

# Print top features
mean_shap = pd.Series(
    np.abs(shap_values).mean(axis=0),
    index=feature_cols
).sort_values(ascending=False)
print('\n🏆 Top 10 Most Important Features (SHAP):')
for i, (feat, val) in enumerate(mean_shap.head(10).items(), 1):
    print(f'   {i:2d}. {feat:<30} SHAP = {val/1e6:.3f}M PKR')

✅ SHAP Global Importance plot saved → eda_plots/13_shap_global_importance.png

🏆 Top 10 Most Important Features (SHAP):
    1. rolling_30_mean_debit          SHAP = 9.211M PKR
    2. rolling_7_mean_debit           SHAP = 3.609M PKR
    3. Daily_Txn_Count                SHAP = 3.558M PKR
    4. Branch_Total_Debit             SHAP = 3.080M PKR
    5. Day                            SHAP = 2.487M PKR
    6. Weekday                        SHAP = 1.974M PKR
    7. Weekday_Sin                    SHAP = 1.601M PKR
    8. lag_1_debit                    SHAP = 1.151M PKR
    9. lag_7_debit                    SHAP = 0.981M PKR
   10. lag_30_debit                   SHAP = 0.959M PKR


---
## Step 5 — SHAP Beeswarm Plot

**Har feature ki value kitni aur kis direction mein prediction ko affect karti hai?**

- Red = High feature value
- Blue = Low feature value
- Right of 0 = Increases prediction (more cash needed)
- Left of 0 = Decreases prediction (less cash needed)

In [5]:
fig, ax = plt.subplots(figsize=(11, 8))
shap.summary_plot(
    shap_values, X_sample,
    feature_names=feature_cols,
    show=False,
    max_display=15
)
plt.title('SHAP Beeswarm Plot \u2014 Feature Impact Direction & Magnitude',
          fontsize=12, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('eda_plots/14_shap_beeswarm.png', dpi=150, bbox_inches='tight')
plt.close()
print('✅ SHAP Beeswarm plot saved → eda_plots/14_shap_beeswarm.png')

✅ SHAP Beeswarm plot saved → eda_plots/14_shap_beeswarm.png


---
## Step 6 — SHAP Waterfall Plot (Single Prediction Explanation)

**Ek specific prediction explain karna — model ne 'X amount' kyun predict kiya?**

Yeh woh plot hai jo manager ko dikhayenge:
> *'Branch 104 ko kal itna cash chahiye... kyunki...'*

In [6]:
# Pick a high-prediction example (most interesting to explain)
y_pred_sample = model.predict(X_sample)
high_idx      = int(np.argmax(y_pred_sample))  # index with highest predicted cash

# Branch and date info
branch_id   = test_meta.iloc[high_idx]['tran_br_code'] if high_idx < len(test_meta) else 'N/A'
pred_amount = y_pred_sample[high_idx] / 1e6
actual_amt  = y_test.iloc[high_idx] / 1e6 if high_idx < len(y_test) else 0

print(f'🔍 Explaining prediction #{high_idx}:')
print(f'   Branch       : {branch_id}')
print(f'   Predicted    : PKR {pred_amount:.2f} Million')
print(f'   Actual       : PKR {actual_amt:.2f} Million')

fig, ax = plt.subplots(figsize=(12, 6))
shap.waterfall_plot(explanation[high_idx], max_display=15, show=False)
plt.title(f'SHAP Waterfall \u2014 Branch {branch_id}: Why PKR {pred_amount:.1f}M Predicted?',
          fontsize=12, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('eda_plots/15_shap_waterfall.png', dpi=150, bbox_inches='tight')
plt.close()
print(f'\n✅ SHAP Waterfall saved → eda_plots/15_shap_waterfall.png')

🔍 Explaining prediction #136:
   Branch       : 234
   Predicted    : PKR 175.36 Million
   Actual       : PKR 150.95 Million



✅ SHAP Waterfall saved → eda_plots/15_shap_waterfall.png


---
## Step 7 — SHAP Dependence Plot (lag_1 vs Prediction)

**Kal ka withdrawal jitna zyada hoga, aaj ka model prediction kitna affect hota hai?**

In [7]:
top_feature = mean_shap.index[0]  # Most important feature

fig, ax = plt.subplots(figsize=(10, 6))
shap.dependence_plot(
    top_feature,
    shap_values,
    X_sample,
    feature_names=feature_cols,
    ax=ax,
    show=False
)
ax.set_title(f'SHAP Dependence Plot \u2014 {top_feature} vs Its SHAP Impact',
             fontsize=12, fontweight='bold')
ax.set_xlabel(f'{top_feature} (PKR)')
ax.set_ylabel('SHAP Value (Impact on Cash Prediction)')
plt.tight_layout()
plt.savefig('eda_plots/16_shap_dependence.png', dpi=150, bbox_inches='tight')
plt.close()
print(f'✅ SHAP Dependence plot saved → eda_plots/16_shap_dependence.png')
print(f'   Feature shown: {top_feature}')

✅ SHAP Dependence plot saved → eda_plots/16_shap_dependence.png
   Feature shown: rolling_30_mean_debit


---
## Step 8 — Branch-Level SHAP Summary

**Per branch — average SHAP impact of top features**

In [8]:
# Attach branch info to sample
X_sample_meta = X_sample.copy()
X_sample_meta['shap_sum'] = np.abs(shap_values).sum(axis=1)

# Top 5 features SHAP per branch
top5 = mean_shap.head(5).index.tolist()
shap_df = pd.DataFrame(shap_values, columns=feature_cols)
shap_df['tran_br_code'] = X_sample['tran_br_code'].values

branch_shap = shap_df.groupby('tran_br_code')[top5].mean() / 1e6

fig, ax = plt.subplots(figsize=(14, 6))
branch_shap.plot(kind='bar', ax=ax, edgecolor='white', width=0.7)
ax.set_title('Average SHAP Impact per Branch \u2014 Top 5 Features',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Branch Code')
ax.set_ylabel('Avg SHAP Value (Million PKR)')
ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax.legend(title='Features', fontsize=8, loc='upper right')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('eda_plots/17_shap_branch_summary.png', dpi=150, bbox_inches='tight')
plt.close()
print('✅ Branch SHAP summary saved → eda_plots/17_shap_branch_summary.png')
print('\n📊 Average SHAP values per branch (Million PKR):')
print(branch_shap.round(2).to_string())

✅ Branch SHAP summary saved → eda_plots/17_shap_branch_summary.png

📊 Average SHAP values per branch (Million PKR):
              rolling_30_mean_debit  rolling_7_mean_debit  Daily_Txn_Count  Branch_Total_Debit  Day
tran_br_code                                                                                       
104                           43.89                 13.91             2.59                4.63 2.02
202                           -5.45                 -1.79            -0.81                0.56 0.81
234                           -2.12                 -0.97             1.90                0.19 0.54
287                          -12.30                 -3.94            -2.42               -4.68 0.76
372                           -9.27                 -3.54            -0.87               -0.66 0.73
394                          -11.73                 -3.37            -1.62               -3.41 0.76
511                          -10.23                 -4.37             1.54          

---
## Step 9 — Save SHAP Values

In [9]:
# Save SHAP values as CSV for future use
shap_df_save = pd.DataFrame(shap_values, columns=feature_cols)
shap_df_save.to_csv('models/shap_values.csv', index=False)

# Save expected value
with open('models/shap_expected_value.json', 'w') as f:
    json.dump({'expected_value': float(explainer.expected_value)}, f)

print('✅ SHAP values saved:')
print('   📄 models/shap_values.csv         — SHAP values for 500 test samples')
print('   📄 models/shap_expected_value.json — Base prediction value')
print(f'\n🎉 Phase 5 COMPLETE!')
print(f'   ➡️  Next: Phase 6 — Evaluation & Reporting')

✅ SHAP values saved:
   📄 models/shap_values.csv         — SHAP values for 500 test samples
   📄 models/shap_expected_value.json — Base prediction value

🎉 Phase 5 COMPLETE!
   ➡️  Next: Phase 6 — Evaluation & Reporting


---
## ✅ Phase 5 Summary

| Plot | Description |
|---|---|
| `13_shap_global_importance.png` | Global feature importance (bar) |
| `14_shap_beeswarm.png` | Feature impact direction & magnitude |
| `15_shap_waterfall.png` | Single prediction explanation |
| `16_shap_dependence.png` | Top feature vs SHAP impact |
| `17_shap_branch_summary.png` | Per-branch SHAP breakdown |

### ➡️ Next: Phase 6 — Evaluation & Reporting